# Expectations about inflation, house prices, and the stock market

TECH2 term paper (H25), analyzing US Survey of Consumer Expectations (SCE) data for Norges Bank.

**AI usage statement:** This notebook was developed with the assistance of an AI coding agent (Claude Code), used to help structure the analysis, write and review pandas/matplotlib code, and draft explanations. All code was reviewed and run by the author.

## Part 1 — Reading in the data

We read all monthly SCE survey files from the `data/` folder and merge them into a single `DataFrame`.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path('data')

In [2]:
def load_sce_data(data_dir):
    """Read and merge all SCE survey wave files into a single DataFrame.

    Parameters
    ----------
    data_dir : Path
        Directory containing the ``SCE-MMM-YYYY.csv`` wave files.

    Returns
    -------
    DataFrame
        Combined individual-level survey data from all waves.
    """
    # SCE files are semicolon-delimited, unlike the comma-delimited CPI data.
    files = sorted(data_dir.glob('SCE-*.csv'))
    waves = (pd.read_csv(f, sep=';', parse_dates=['date']) for f in files)
    return pd.concat(waves, ignore_index=True)

In [3]:
def report_sample_stats(df, label):
    """Print sample-size statistics for a survey DataFrame.

    Parameters
    ----------
    df : DataFrame
        Individual-level survey data with ``userid``, ``wid``, and ``date``
        columns.
    label : str
        Description of the dataset, used in the printed report.
    """
    n_individuals = df['userid'].nunique()
    n_observations = len(df)
    n_waves = df['wid'].nunique()
    first_date = df['date'].min()
    last_date = df['date'].max()

    print(f'{label}')
    print(f'  Unique individuals: {n_individuals}')
    print(f'  Observations (rows): {n_observations}')
    print(f'  Unique survey waves: {n_waves}')
    print(f'  First date observed: {first_date.date()}')
    print(f'  Last date observed: {last_date.date()}')

In [4]:
sce = load_sce_data(DATA_DIR)

In [5]:
report_sample_stats(sce, 'Raw sample')

Raw sample
  Unique individuals: 23369
  Observations (rows): 176101
  Unique survey waves: 139
  First date observed: 2013-06-01
  Last date observed: 2024-12-31


## Part 2 — Data pre-processing


### Fill missing numeracy responses

The numeracy questions are only asked in a respondent's first wave, so we carry those answers forward to later waves for the same individual.


In [6]:
DEMOGRAPHIC_COLS = ['female', 'age', 'educ']
EXPECTATION_COLS = ['inflation', 'house_price_change', 'prob_stocks_up']
NUMERACY_COLS = [
    'num_lit_q1_correct',
    'num_lit_q2_correct',
    'num_lit_q3_correct',
    'num_lit_q5_correct',
    'num_lit_q6_correct',
    'num_lit_q8_correct',
    'num_lit_q9_correct',
]

In [7]:
def fill_numeracy(df, cols):
    """Fill missing numeracy indicators using each individual's first observation.

    The numeracy questions are only asked the first time a respondent enters
    the survey, so later waves have missing values in ``cols``. This fills
    them in using the value from each individual's earliest survey date.

    Parameters
    ----------
    df : DataFrame
        Individual-level survey data, with one row per respondent per wave.
    cols : list of str
        Numeracy indicator columns to fill.

    Returns
    -------
    DataFrame
        Copy of ``df`` with ``cols`` filled within each ``userid`` group.
    """
    df = df.sort_values(['userid', 'date'])
    df[cols] = df.groupby('userid')[cols].transform('first')
    return df

In [8]:
sce = fill_numeracy(sce, NUMERACY_COLS)

### Drop missing observations

We drop rows missing demographic information, expectations responses, or numeracy responses (after the fill above), reporting the number of observations dropped at each step.


In [9]:
def drop_missing(df, cols, label):
    """Drop rows with missing values in ``cols`` and report how many were dropped.

    Parameters
    ----------
    df : DataFrame
        Data to filter.
    cols : list of str
        Columns checked for missing values.
    label : str
        Description of this step, used in the printed report.

    Returns
    -------
    DataFrame
        Subset of ``df`` with no missing values in ``cols``.
    """
    before = len(df)
    df = df.dropna(subset=cols)
    print(f'{label}: dropped {before - len(df)} observations ({len(df)} remaining)')
    return df

In [10]:
sce = drop_missing(sce, DEMOGRAPHIC_COLS, 'Missing demographics')
sce = drop_missing(sce, EXPECTATION_COLS, 'Missing expectations')
sce = drop_missing(sce, NUMERACY_COLS, 'Missing numeracy (after fill)')

Missing demographics: dropped 868 observations (175233 remaining)
Missing expectations: dropped 1683 observations (173550 remaining)
Missing numeracy (after fill): dropped 35974 observations (137576 remaining)


### Drop outliers

For each expectations variable, we drop observations below the 0.1th percentile and above the 99.9th percentile, reporting the number dropped at each bound.


In [11]:
def drop_outliers(df, col):
    """Drop observations with extreme values in ``col`` based on percentiles.

    Observations below the 0.1th percentile or above the 99.9th percentile
    are dropped, with each bound reported separately.

    Parameters
    ----------
    df : DataFrame
        Data to filter.
    col : str
        Expectations column to trim.

    Returns
    -------
    DataFrame
        Subset of ``df`` with outliers in ``col`` removed.
    """
    lower, upper = df[col].quantile([0.001, 0.999])

    before = len(df)
    df = df[df[col] >= lower]
    print(f'{col}: dropped {before - len(df)} observations below {lower:.2f}')

    before = len(df)
    df = df[df[col] <= upper]
    print(f'{col}: dropped {before - len(df)} observations above {upper:.2f}')

    return df

In [12]:
for col in EXPECTATION_COLS:
    sce = drop_outliers(sce, col)

inflation: dropped 92 observations below -75.00
inflation: dropped 27 observations above 100.00
house_price_change: dropped 133 observations below -50.00
house_price_change: dropped 46 observations above 100.00


prob_stocks_up: dropped 0 observations below 0.00
prob_stocks_up: dropped 0 observations above 100.00


### Derived variables: college and numerical literacy

`college` flags respondents with a bachelor's degree or higher. `num_lit_high` flags respondents whose total numeracy score is above the sample median.


In [13]:
sce['college'] = (sce['educ'] == 4).astype(int)

In [14]:
sce['num_lit_total'] = sce[NUMERACY_COLS].sum(axis=1)

# One row per individual, since numeracy indicators are now constant within userid.
numeracy_dist = (
    sce.drop_duplicates('userid')['num_lit_total']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)
print('Share of individuals (%) by number of correct numeracy responses:')
print(numeracy_dist.round(1).to_string())

Share of individuals (%) by number of correct numeracy responses:
num_lit_total
0.0     0.1
1.0     0.7
2.0     2.4
3.0     5.7
4.0    10.5
5.0    16.9
6.0    27.4
7.0    36.2


In [15]:
median_num_lit = sce.drop_duplicates('userid')['num_lit_total'].median()
sce['num_lit_high'] = (sce['num_lit_total'] > median_num_lit).astype(int)

### Final sample size


In [16]:
report_sample_stats(sce, 'Cleaned sample')

Cleaned sample
  Unique individuals: 17701
  Observations (rows): 137278
  Unique survey waves: 117
  First date observed: 2015-04-02
  Last date observed: 2024-12-31
